# Protonation Review — PROPKA vs Literature

**Runs on CESGA via `juplaunch`**  (memory `[[reference_juplaunch]]`).

## Connection setup

On your **laptop** before launching this notebook:

```bash
# 1. Start PyMOL with the RPC server on the laptop
export PYMOL_PATH=/usr/lib/python3/dist-packages/pymol   # Debian-fix per feedback_pymol_remote_debian
pymol -R
# → PyMOL RPC listening on :9123

# 2. Set up the reverse tunnel to CESGA
ssh -R 9123:localhost:9123 ft3.cesga.es
```

On CESGA:

```bash
~/bin/juplaunch     # prints the JupyterLab URL to paste into your laptop browser
```

Then open this notebook in the browser and run the cells top-down.
The `PymolSession(hostname="localhost", port=9123)` call reaches through
the reverse tunnel to the laptop's PyMOL GUI.


*(PyMOL is not required for this notebook — the connection cell is still included so you can quickly load a residue afterwards.)*

## Step 1 — (Optional) connect to PyMOL for cross-checking

In [ ]:
# Connect to laptop PyMOL via reverse tunnel (localhost:9123)
from pymol_remote.client import PymolSession
import sys, pathlib

pm = PymolSession(hostname="localhost", port=9123, timeout=10.0)
print("Connected. Objects currently loaded:", pm.get_names())


## Step 2 — Parsers

In [ ]:
# Parsers: PROPKA .pka files + paper_evidence.md
import pathlib, re
from dataclasses import dataclass, field

DELIVERED_ROOT = pathlib.Path("/mnt/netapp1/Store_othcxlwa/newbench_27/delivered")

_PKA_HEADER_RE = re.compile(r"^\s*Group\s+", re.IGNORECASE)
_PKA_LINE_RE = re.compile(
    r"^\s*"
    r"(?P<resn>[A-Z]{3})\s+"          # residue name
    r"(?P<resi>\-?\d+)\s+"           # residue number
    r"(?P<chain>[A-Za-z0-9])\s+"       # chain id
    r"(?P<pka>\-?\d+\.\d+)"          # pKa
)

@dataclass
class PkaEntry:
    chain: str
    resi: int
    resn: str
    pka: float

def parse_pka_file(path: pathlib.Path) -> list[PkaEntry]:
    if not path.is_file():
        return []
    out = []
    for line in path.read_text().splitlines():
        m = _PKA_LINE_RE.match(line)
        if m:
            try:
                out.append(PkaEntry(
                    chain=m.group("chain"), resi=int(m.group("resi")),
                    resn=m.group("resn"),   pka=float(m.group("pka")),
                ))
            except (ValueError, KeyError):
                continue
    return out

def merge_propka_dir(protonation_dir: pathlib.Path) -> dict[tuple[str,int], list[PkaEntry]]:
    out: dict[tuple[str,int], list[PkaEntry]] = {}
    if not protonation_dir.is_dir():
        return out
    for pka_file in sorted(protonation_dir.glob("*.pka")):
        for e in parse_pka_file(pka_file):
            out.setdefault((e.chain, e.resi), []).append(e)
    return out


_RESI_HEADING_RE = re.compile(r"^###\s+([A-Z]{3})(\d+)\s*$", re.IGNORECASE)
_QUOTE_LINE_RE = re.compile(r"^>\s*(.*)$")

@dataclass
class PaperNote:
    resn: str
    resi: int
    quote: str

def parse_paper_evidence(md_path: pathlib.Path) -> list[PaperNote]:
    if not md_path.is_file():
        return []
    text = md_path.read_text()
    notes: list[PaperNote] = []
    lines = text.splitlines()
    i = 0
    while i < len(lines):
        m = _RESI_HEADING_RE.match(lines[i].strip())
        if m:
            resn, resi = m.group(1).upper(), int(m.group(2))
            j = i + 1
            quotes = []
            while j < len(lines) and not _RESI_HEADING_RE.match(lines[j].strip()) \
                                  and not lines[j].startswith("## "):
                q = _QUOTE_LINE_RE.match(lines[j])
                if q:
                    quotes.append(q.group(1).strip())
                j += 1
            i = j
            if quotes:
                notes.append(PaperNote(resn=resn, resi=resi,
                                       quote=" ".join(quotes)[:400]))
            continue
        i += 1
    return notes


## Step 3 — Discover usable proteins (need both PROPKA output + `paper_evidence.md`)

In [ ]:
# Discover proteins that have both PROPKA output and paper_evidence.md
pdb_ids = sorted(p.name for p in DELIVERED_ROOT.iterdir() if p.is_dir())
usable = []
for pid in pdb_ids:
    entry = DELIVERED_ROOT / pid
    has_propka = (entry / "protonation").is_dir() \
                 and any((entry / "protonation").glob("*.pka"))
    has_paper = (entry / "paper" / "paper_evidence.md").is_file()
    if has_propka and has_paper:
        usable.append(pid)
print(f"{len(usable)} of {len(pdb_ids)} delivered proteins have both PROPKA + paper_evidence:")
print(" ".join(usable))


## Step 4 — Render side-by-side

In [ ]:
# Side-by-side renderer: for one PDB, PROPKA table (top) + paper snippets (bottom)
from IPython.display import Markdown, display

def render_protonation_side_by_side(pdb_id: str):
    entry = DELIVERED_ROOT / pdb_id
    propka = merge_propka_dir(entry / "protonation")
    paper = parse_paper_evidence(entry / "paper" / "paper_evidence.md")

    md_lines = [f"## {pdb_id} — PROPKA vs Literature", ""]

    # Union of residues from both sources
    propka_keys = set(propka.keys())
    paper_keys = {(("A"), p.resi) for p in paper}  # paper_evidence has no chain, assume A
    all_keys = sorted(propka_keys | paper_keys, key=lambda k: (k[0], k[1]))

    md_lines.append("| Chain | Resi | Resn | PROPKA pKa (min–max) | Paper snippet |")
    md_lines.append("|---|---:|---|---|---|")
    paper_by_resi = {p.resi: p for p in paper}
    for (chain, resi) in all_keys:
        pkas = propka.get((chain, resi), [])
        resn = pkas[0].resn if pkas else (paper_by_resi[resi].resn if resi in paper_by_resi else "—")
        if pkas:
            pks = [f"{p.pka:.2f}" for p in pkas]
            pka_str = pks[0] if len(pks) == 1 else f"{min(p.pka for p in pkas):.2f}–{max(p.pka for p in pkas):.2f}"
        else:
            pka_str = "—"
        snippet = paper_by_resi[resi].quote if resi in paper_by_resi else "—"
        snippet = snippet.replace("|", "\\|")[:120] + ("…" if resi in paper_by_resi and len(paper_by_resi[resi].quote) > 120 else "")
        md_lines.append(f"| {chain} | {resi} | {resn} | {pka_str} | {snippet} |")

    display(Markdown("\n".join(md_lines)))

# Try the first available one:
if usable:
    render_protonation_side_by_side(usable[0])


## Step 5 — Interactive selector

In [ ]:
import ipywidgets as W
from IPython.display import display, clear_output

if usable:
    dropdown = W.Dropdown(options=usable, description="PDB:", layout={"width": "260px"})
    btn = W.Button(description="Render", button_style="primary")
    out = W.Output()

    def _on_click(_):
        with out:
            clear_output()
            render_protonation_side_by_side(dropdown.value)

    btn.on_click(_on_click)
    display(W.HBox([dropdown, btn]), out)
else:
    print("No proteins have both PROPKA + paper_evidence yet.")


## Step 6 — Show the pre-generated `active_site_protonation.md`

This is the shipped summary FRUTON already wrote per protein.

In [ ]:
# Optional: display the pre-generated active_site_protonation.md for one protein
def show_active_site_md(pdb_id: str):
    p = DELIVERED_ROOT / pdb_id / "active_site_protonation.md"
    if not p.is_file():
        print(f"No active_site_protonation.md for {pdb_id}")
        return
    display(Markdown(p.read_text()))

if usable:
    show_active_site_md(usable[0])
